[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/visual-nlp-workshop/blob/master/databricks/VisualNLP_Cluster_Setup.ipynb)

# Configure Existing Databricks Cluster for John Snow Labs OCR / Healthcare

Follow the cells in order. Do not run the cluster edit or library install cells until the workspace URL, token, cluster ID, runtime version, volume path, and JSL license JSON have been verified.

## Imports

This notebook configures an existing Databricks cluster for John Snow Labs OCR / Healthcare libraries. Before continuing, make sure you have a Databricks workspace URL, personal access token, target cluster ID, Unity Catalog volume path, and John Snow Labs license JSON available.

In [1]:
from pathlib import Path
import requests
import json

## Required User Configuration

### Find the Databricks workspace URL and cluster ID

Open the cluster's **Configuration** page and check the browser address bar:

```text
https://adb-1234567890123456.7.azuredatabricks.net/compute/clusters/0123-456789-abcde123/configuration
|_________________________________________________|                |___________________|
                  Workspace URL                                              Cluster ID
```

### Configuration parameters

| Parameter | Supported value/example | Description |
|---|---|---|
| **Workspace URL** | `https://adb-1234567890123456.7.azuredatabricks.net` | Base URL of the Databricks workspace |
| **Cluster ID** | `0123-456789-abcde123` | ID of the cluster to configure or start |
| **Cluster Type** | `cpu` or `gpu` | Type of cluster to create |
| **Runtime Version** | `15.4 LTS` | Supported Databricks Runtime version |
| **Volume Root Directory** | `/Volumes/source/files` | Directory where the JSL JAR and wheel files will be stored |
| **Private** | `True` or `False` | Controls whether sensitive logging is allowed |

> Replace all example values with the values from your Databricks environment.

In [19]:
workspace_url = ""
token = ""
cluster_id = ""

# Cluster Type [ cpu, gpu ]
cluster_type = "gpu"

# Cluster Runtime Version [ 15.4, 16.4 ]
cluster_runtime_version = "16.4"

# Should not contain trailing "/"
volume_root_dir = "/Volumes/visual_nlp/default/visual_nlp/packages"

# Hide JSL and Databricks Secrets while logging
private = True

## License JSON

Before running this cell, upload or place the John Snow Labs license JSON at the path assigned to `license`. If you are running outside Colab, replace `/content/...` with the path used by your notebook environment.

In [8]:
license = "/content/spark_nlp_for_healthcare_spark_ocr_xxxx.json"

if license and "json" in license:

    with open(license, "r") as creds_in:
        creds = json.loads(creds_in.read())

        for key in creds.keys():

            if private:
              value = "*" * min(10, len(creds[key]))
            else:
              value = creds[key]

            print(f"{key}: {value}")
else:
    raise Exception("License JSON File is not specified")

SPARK_NLP_LICENSE: **********
SECRET: **********
JSL_VERSION: *****
SPARK_OCR_LICENSE: **********
SPARK_OCR_SECRET: **********
OCR_VERSION: *****
PUBLIC_VERSION: *****
AWS_ACCESS_KEY_ID: **********
AWS_SECRET_ACCESS_KEY: **********
AWS_SESSION_TOKEN: **********


## Runtime Package Matrix

Review `DB_RUNTIME` only if your cluster runtime is different. The package pins here are runtime-specific and should match the Databricks Runtime selected in the configuration cell below.

In [9]:
pypi_packages_15_4_lts = [
    "scikit-image==0.21.0",
    "scipy==1.13.0",
    "numpy==2.0.0",
    "pandas==2.2.3",
    "PyArrow==23.0.0",
    "matplotlib==3.10.8"
]

pypi_packages_16_4_lts = [
    "numpy==2.0.0",
    "scipy==1.13.0",
    "pandas==2.2.3",
    "PyArrow==23.0.0",
    "matplotlib==3.10.8",
    "scikit-image>=0.25.2",
    "pillow>=10.1",
    "opencv-python==4.12.0.88",
    "pyspark==3.5.2",
    "python-gdcm==3.0.26"
]

DB_RUNTIME = { "15.4" : {
    "python" : pypi_packages_15_4_lts,
    "cpu" : {"maven" : f"com.johnsnowlabs.nlp:spark-nlp_2.12:{creds["PUBLIC_VERSION"]}", "exclusion" : "com.microsoft.onnxruntime:onnxruntime", "onnx" : "com.microsoft.onnxruntime:onnxruntime:1.18.0"},
    "gpu" : {"maven" : f"com.johnsnowlabs.nlp:spark-nlp-gpu_2.12:{creds["PUBLIC_VERSION"]}", "exclusion" : "com.microsoft.onnxruntime:onnxruntime_gpu", "onnx" : "com.microsoft.onnxruntime:onnxruntime_gpu:1.18.0"}
  },
               "16.4" : {
    "python" : pypi_packages_16_4_lts,
    "cpu" : {"maven" : f"com.johnsnowlabs.nlp:spark-nlp_2.12:{creds["PUBLIC_VERSION"]}", "exclusion" : None, "onnx" : None},
    "gpu" : {"maven" : f"com.johnsnowlabs.nlp:spark-nlp-gpu_2.12:{creds["PUBLIC_VERSION"]}", "exclusion" : None, "onnx" : None}
  }
}

## Databricks API Authentication

This cell prepares Databricks API authentication. Run it only after confirming the token and cluster ID above are populated.

In [10]:
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

params = {"cluster_id": cluster_id}

## Cluster Existence Check

Validation checkpoint: run this cell to confirm the target cluster exists and that the token has permission to read it. Stop if this does not print `Cluster exists`.

In [11]:
response = requests.get(
    f"{workspace_url}/api/2.0/clusters/get",
    headers=headers,
    params=params,
    timeout=30,
)

if response.status_code == 200:
    cluster = response.json()
    print("Cluster exists")
    print("Name:", "*" * len(cluster["cluster_name"]) if private else cluster["cluster_name"])
    print("State:", cluster["state"])
elif response.status_code == 404:
    print("Cluster does not exist")
else:
    print("Request failed:", response.status_code, response.text)

Cluster exists
Name: ****
State: TERMINATED


## Existing Cluster Settings

Review the existing Spark configuration and environment variables printed here. If the cluster has important custom settings, confirm they are preserved before proceeding.

In [12]:
# Retrieve current cluster configuration
response = requests.get(
    f"{workspace_url}/api/2.0/clusters/get",
    headers=headers,
    params=params,
    timeout=30,
)

response.raise_for_status()

cluster = response.json()

spark_conf = dict(cluster.get("spark_conf") or {})
spark_env_vars = dict(cluster.get("spark_env_vars") or {})

print(f"Existing Spark Conf: {spark_conf}")
print(f"Existing Environment Variables: {spark_env_vars}")

Existing Spark Conf: {}
Existing Environment Variables: {}


## Spark Configuration Update

In [13]:
spark_conf.update({
    "spark.sql.legacy.allowUntypedScalaUDF": "true"
})

## JSL Environment Variables

Configure the environment variables required for your license type before running the notebook.

In this notebook, only `SPARK_OCR_LICENSE` is configured. Depending on your license and deployment requirements, you can configure additional variables:

- `SPARK_OCR_LICENSE`
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_SESSION_TOKEN`

In [14]:
spark_env_vars.update({
    "SPARK_OCR_LICENSE": creds["SPARK_OCR_LICENSE"]
})

## Edit Cluster Configuration

This cell edits the Databricks cluster configuration. Run it only after confirming the cluster ID, runtime version, Spark configuration, and license environment variables are correct. A cluster restart may be needed before these changes take effect.

In [15]:
# Fields that are valid cluster-edit settings and should be preserved from GET.
editable_fields = [
    "cluster_name",
    "spark_version",
    "node_type_id",
    "driver_node_type_id",
    "instance_pool_id",
    "driver_instance_pool_id",
    "autoscale",
    "num_workers",
    "virtual_cluster_size",
    "autotermination_minutes",
    "enable_elastic_disk",
    "cluster_log_conf",
    "init_scripts",
    "custom_tags",
    "spark_env_vars",
    "spark_conf",
    "ssh_public_keys",
    "aws_attributes",
    "azure_attributes",
    "gcp_attributes",
    "data_security_mode",
    "single_user_name",
    "runtime_engine",
    "workload_type",
    "docker_image",
    "remote_disk_throughput",
    "remote_disk_volume_type",
    "policy_id",
]

payload = {
    key: cluster[key]
    for key in editable_fields
    if key in cluster and cluster[key] is not None
}

payload["cluster_id"] = cluster_id
payload["spark_conf"] = spark_conf
payload["spark_env_vars"] = spark_env_vars

for key in ("autoscale", "num_workers", "virtual_cluster_size"):
    if key != next(
        k for k in ("autoscale", "num_workers", "virtual_cluster_size")
        if cluster.get(k) is not None
    ):
        payload.pop(key, None)

response = requests.post(
    f"{workspace_url}/api/2.0/clusters/edit",
    headers={**headers, "Content-Type": "application/json"},
    json=payload,
    timeout=30,
)

print(response.status_code)
print(response.text)
response.raise_for_status()

200
{}


## Download Helper

Helper function only. It downloads library artifacts locally into `./Downloads`; no Databricks changes happen in this cell.

In [16]:
def download_file(url, output_dir = "./Downloads"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = url.split("?")[0].rsplit("/", 1)[-1]
    output_path = output_dir / filename

    with requests.get(url, stream=True, timeout=1800) as response:
        response.raise_for_status()

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    return str(output_path.resolve())

## Download JSL Artifacts

This downloads private JSL wheel and JAR artifacts using secrets from the license JSON. Confirm your environment can access `pypi.johnsnowlabs.com`; downloads may take several minutes.

In [17]:
jsl_resources = [
    # Visual
    f'https://pypi.johnsnowlabs.com/{creds["SPARK_OCR_SECRET"]}/spark-ocr/spark_ocr-{creds["OCR_VERSION"]}-py3-none-any.whl',
    f'https://pypi.johnsnowlabs.com/{creds["SPARK_OCR_SECRET"]}/jars/spark-ocr-assembly-{creds["OCR_VERSION"]}.jar',

    # Healthcare
    f'https://pypi.johnsnowlabs.com/{creds["SECRET"]}/spark-nlp-jsl/spark_nlp_jsl-{creds["JSL_VERSION"]}-py3-none-any.whl',
    f'https://pypi.johnsnowlabs.com/{creds["SECRET"]}/spark-nlp-jsl-{creds["JSL_VERSION"]}.jar'
]

downloaded_files = [download_file(url) for url in jsl_resources]

downloaded_files

['/content/Downloads/spark_ocr-6.4.2-py3-none-any.whl',
 '/content/Downloads/spark-ocr-assembly-6.4.2.jar',
 '/content/Downloads/spark_nlp_jsl-6.4.1-py3-none-any.whl',
 '/content/Downloads/spark-nlp-jsl-6.4.1.jar']

## Install Maven Libraries

This requests Maven library installation on the cluster, including Spark NLP GPU and ONNX Runtime. Check the response for success before continuing.

In [22]:
runtime_cfg = DB_RUNTIME[cluster_runtime_version][cluster_type]

main_maven = {"coordinates": runtime_cfg["maven"]}

if runtime_cfg.get("exclusion"):
    main_maven["exclusions"] = [runtime_cfg["exclusion"]]

maven_libraries = [{"maven": main_maven}]

if runtime_cfg.get("onnx"):
    maven_libraries.append({"maven": {"coordinates": runtime_cfg["onnx"]}})

response = requests.post(
    f"{workspace_url.rstrip('/')}/api/2.0/libraries/install",
    headers=headers,
    params=params,
    json={
        "cluster_id": cluster_id,
        "libraries": maven_libraries,
    },
    timeout=30,
)

print("Status:", response.status_code)
print("Response:", response.text)
response.raise_for_status()

print("Maven library installation requested")

Status: 200
Response: {}
Maven library installation requested


In [23]:
maven_libraries

[{'maven': {'coordinates': 'com.johnsnowlabs.nlp:spark-nlp-gpu_2.12:6.4.2'}}]

## Upload Artifacts to Databricks Volume

This creates the target Databricks volume directory and uploads downloaded `.whl` and `.jar` files. Confirm `volume_root_dir` points to the intended Unity Catalog volume before running.

In [24]:
from pathlib import Path
from urllib.parse import quote
import requests

# Create the destination directory if needed
encoded_dir = quote(volume_root_dir, safe="/")

response = requests.put(
    f"{workspace_url.rstrip('/')}/api/2.0/fs/directories{encoded_dir}",
    headers=headers,
    timeout=30,
)
response.raise_for_status()

uploaded_paths = []

for local_file in downloaded_files:
    local_path = Path(local_file)
    volume_path = f"{volume_root_dir}/{local_path.name}"
    encoded_path = quote(volume_path, safe="/")

    with local_path.open("rb") as file:
        response = requests.put(
            f"{workspace_url.rstrip('/')}/api/2.0/fs/files{encoded_path}",
            headers={
                **headers,
                "Content-Type": "application/octet-stream",
            },
            params={"overwrite": "true"},
            data=file,
            timeout=1800,
        )

    response.raise_for_status()
    uploaded_paths.append(volume_path)
    print(f"Uploaded: {volume_path}")

Uploaded: /Volumes/visual_nlp/default/visual_nlp/packages/spark_ocr-6.4.2-py3-none-any.whl
Uploaded: /Volumes/visual_nlp/default/visual_nlp/packages/spark-ocr-assembly-6.4.2.jar
Uploaded: /Volumes/visual_nlp/default/visual_nlp/packages/spark_nlp_jsl-6.4.1-py3-none-any.whl
Uploaded: /Volumes/visual_nlp/default/visual_nlp/packages/spark-nlp-jsl-6.4.1.jar


## Build Library Install Payload

This builds the final library install payload from uploaded artifacts plus runtime-specific PyPI packages. Review `libraries` in the next cell before installing.

In [25]:
libraries = []

for path in uploaded_paths:
    if path.lower().endswith(".jar"):
        libraries.append({"jar": path})
    elif path.lower().endswith(".whl"):
        libraries.append({"whl": path})
    else:
        print(f"Skipping unsupported file: {path}")

libraries.extend([
    {
        "pypi": {
            "package": package
        }
    }
    for package in DB_RUNTIME[cluster_runtime_version]["python"]
])

## Review Libraries

Manual review checkpoint: inspect the generated library list. Continue only if all expected `.jar`, `.whl`, Maven, and PyPI packages are present.

In [26]:
libraries

[{'whl': '/Volumes/visual_nlp/default/visual_nlp/packages/spark_ocr-6.4.2-py3-none-any.whl'},
 {'jar': '/Volumes/visual_nlp/default/visual_nlp/packages/spark-ocr-assembly-6.4.2.jar'},
 {'whl': '/Volumes/visual_nlp/default/visual_nlp/packages/spark_nlp_jsl-6.4.1-py3-none-any.whl'},
 {'jar': '/Volumes/visual_nlp/default/visual_nlp/packages/spark-nlp-jsl-6.4.1.jar'},
 {'pypi': {'package': 'numpy==2.0.0'}},
 {'pypi': {'package': 'scipy==1.13.0'}},
 {'pypi': {'package': 'pandas==2.2.3'}},
 {'pypi': {'package': 'PyArrow==23.0.0'}},
 {'pypi': {'package': 'matplotlib==3.10.8'}},
 {'pypi': {'package': 'scikit-image>=0.25.2'}},
 {'pypi': {'package': 'pillow>=10.1'}},
 {'pypi': {'package': 'opencv-python==4.12.0.88'}},
 {'pypi': {'package': 'pyspark==3.5.2'}},
 {'pypi': {'package': 'python-gdcm==3.0.26'}}]

## Install Prepared Libraries

This installs the prepared libraries on the cluster. After completion, verify Databricks library status and restart the cluster if required by your environment.

In [27]:
payload = {
    "cluster_id": cluster_id,
    "libraries": libraries,
}

response = requests.post(
    f"{workspace_url.rstrip('/')}/api/2.0/libraries/install",
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    },
    json=payload,
    timeout=30,
)

print("Status:", response.status_code)
print("Response:", response.text)
response.raise_for_status()

Status: 200
Response: {}


# Start Cluster From the UI
